# Harbour Surveillance - Standalone T2-T5 Tracking Solution

This notebook uses the scenario JSON files generated by the provided school notebook in `harbour_sim_output/`. The original `harbour_simulation.ipynb` is not modified.

Implemented here:
- T2 coordinate frame manager tests.
- T3 radar-only EKF on Scenario A.
- T4 radar + stereo camera fusion on Scenario B, sequential and joint.
- T5 radar + camera + AIS asynchronous fusion on Scenario C, including AIS dropout.

## 0. Setup

In [1]:
from pathlib import Path
import json
import sys

import numpy as np

ROOT = Path.cwd()
if ROOT.name == "tracking_solution":
    ROOT = ROOT.parent

TRACKING_DIR = ROOT / "tracking_solution"
SCENARIO_DIR = ROOT / "harbour_sim_output"
sys.path.insert(0, str(TRACKING_DIR))

from harbour_tracking import (
    ScenarioData,
    first_accepted_after,
    max_history_gap,
    rmse,
    run_tracking,
)
from run_t2_t5 import run_t2_unit_tests, result_summary

print(f"Repository root: {ROOT}")
print(f"Scenario data:   {SCENARIO_DIR}")

Repository root: c:\Users\kryst\Documents\GitHub\Multi-Sensor-Multi-Target-Tracking-System-for-Harbour-Surveillance
Scenario data:   c:\Users\kryst\Documents\GitHub\Multi-Sensor-Multi-Target-Tracking-System-for-Harbour-Surveillance\harbour_sim_output


## 1. Load School Scenario Files

In [2]:
scenario_a = ScenarioData.load(SCENARIO_DIR / "scenario_A.json")
scenario_b = ScenarioData.load(SCENARIO_DIR / "scenario_B.json")
scenario_c = ScenarioData.load(SCENARIO_DIR / "scenario_C.json")

for scenario in (scenario_a, scenario_b, scenario_c):
    counts = {}
    for measurement in scenario.measurements:
        counts[measurement.sensor_id] = counts.get(measurement.sensor_id, 0) + 1
    print(
        f"Scenario {scenario.scenario_name}: "
        f"t_end={scenario.t_end:.1f}s, "
        f"targets={len(scenario.ground_truth)}, "
        f"measurements={counts}"
    )

Scenario A: t_end=120.0s, targets=1, measurements={'gnss': 120, 'radar': 143}
Scenario B: t_end=120.0s, targets=1, measurements={'gnss': 120, 'camera': 131, 'radar': 158}
Scenario C: t_end=150.0s, targets=1, measurements={'gnss': 150, 'camera': 197, 'ais': 47, 'radar': 174}


## 2. T2 - Coordinate Frame Manager

In [3]:
t2_results = run_t2_unit_tests()
t2_results

{'radar_basic': 'passed',
 'camera_offset': 'passed',
 'ais_same_origin': 'passed',
 'jacobian_finite_difference': 'passed',
 'ais_covariance_transform': 'passed'}

The AIS update is handled as the assignment requests: AIS NED reports are converted to implied range/bearing relative to the nearest vessel GNSS fix. Because the measurement units change, the AIS Cartesian covariance is transformed through the polar Jacobian before the EKF update.

## 3. T3 - Scenario A, Radar-Only EKF

In [4]:
t3 = run_tracking(
    scenario_a,
    allowed_sensors=("radar",),
    fusion_mode="sequential",
    bootstrap_sensors=("radar",),
    name="T3 radar only",
)

t3_summary = result_summary(t3, scenario_a, start_time=20.0)
t3_confirm_limit = t3.bootstrap_time + 5.0 * (1.0 / 0.3)
t3_pass = {
    "confirmation": t3.confirmation_time is not None and t3.confirmation_time <= t3_confirm_limit,
    "rmse": t3_summary["rmse_after_start_m"] < 12.0,
    "nis": t3_summary["nis_inside_95_pct"] >= 90.0,
}

print(json.dumps(t3_summary, indent=2))
print("pass:", t3_pass)

{
  "bootstrap_sensor": "radar",
  "bootstrap_time_s": 6.6667,
  "confirmation_time_s": 16.6667,
  "accepted_updates": 32,
  "rmse_after_start_m": 3.546,
  "nis_inside_95_pct": 90.62,
  "accepted_by_sensor": {
    "radar": 32,
    "camera": 0,
    "ais": 0
  }
}
pass: {'confirmation': True, 'rmse': True, 'nis': True}


Track initiation is measurement-only. It does not use `target_id` or `is_false_alarm` labels from the simulation JSON.

## 4. T4 - Scenario B, Radar + Stereo Camera

In [5]:
t4_radar = run_tracking(
    scenario_b,
    allowed_sensors=("radar",),
    fusion_mode="sequential",
    bootstrap_sensors=("radar",),
    name="T4 radar baseline",
)
t4_seq = run_tracking(
    scenario_b,
    allowed_sensors=("radar", "camera"),
    fusion_mode="sequential",
    bootstrap_sensors=("radar",),
    name="T4 sequential",
)
t4_joint = run_tracking(
    scenario_b,
    allowed_sensors=("radar", "camera"),
    fusion_mode="joint",
    bootstrap_sensors=("radar",),
    name="T4 joint",
)

t4_summary = {
    "radar_only": result_summary(t4_radar, scenario_b, start_time=5.0),
    "sequential": result_summary(t4_seq, scenario_b, start_time=5.0),
    "joint": result_summary(t4_joint, scenario_b, start_time=5.0),
    "rmse_window_0_20_m": {
        "radar_only": round(rmse(t4_radar, scenario_b, start_time=0.0, end_time=20.0), 3),
        "sequential": round(rmse(t4_seq, scenario_b, start_time=0.0, end_time=20.0), 3),
        "joint": round(rmse(t4_joint, scenario_b, start_time=0.0, end_time=20.0), 3),
    },
}

print(json.dumps(t4_summary, indent=2))

{
  "radar_only": {
    "bootstrap_sensor": "radar",
    "bootstrap_time_s": 6.4,
    "confirmation_time_s": 12.8,
    "accepted_updates": 35,
    "rmse_after_start_m": 3.238,
    "nis_inside_95_pct": 94.29,
    "accepted_by_sensor": {
      "radar": 35,
      "camera": 0,
      "ais": 0
    }
  },
  "sequential": {
    "bootstrap_sensor": "radar",
    "bootstrap_time_s": 3.2,
    "confirmation_time_s": 6.0,
    "accepted_updates": 42,
    "rmse_after_start_m": 2.609,
    "nis_inside_95_pct": 93.02,
    "accepted_by_sensor": {
      "radar": 36,
      "camera": 7,
      "ais": 0
    }
  },
  "joint": {
    "bootstrap_sensor": "radar",
    "bootstrap_time_s": 3.2,
    "confirmation_time_s": 6.0,
    "accepted_updates": 42,
    "rmse_after_start_m": 2.608,
    "nis_inside_95_pct": 92.86,
    "accepted_by_sensor": {
      "radar": 36,
      "camera": 7,
      "ais": 0
    }
  },
  "rmse_window_0_20_m": {
    "radar_only": 6.003,
    "sequential": 4.336,
    "joint": 4.336
  }
}


The camera only has true detections during the early visible part of Scenario B. The local RMSE window `0-20 s` therefore shows the camera benefit most clearly.

## 5. T5 - Scenario C, Add AIS and Handle Dropout

In [6]:
t5_no_ais = run_tracking(
    scenario_c,
    allowed_sensors=("radar", "camera"),
    fusion_mode="sequential",
    bootstrap_sensors=("radar",),
    name="T5 no AIS",
)
t5_with_ais = run_tracking(
    scenario_c,
    allowed_sensors=("radar", "camera", "ais"),
    fusion_mode="sequential",
    bootstrap_sensors=("radar",),
    name="T5 with AIS",
)
t5_ais_only = run_tracking(
    scenario_c,
    allowed_sensors=("ais",),
    fusion_mode="sequential",
    bootstrap_sensors=("ais",),
    name="T5 AIS-only initiation",
)

t5_summary = {
    "without_ais": result_summary(t5_no_ais, scenario_c, start_time=20.0),
    "with_ais": result_summary(t5_with_ais, scenario_c, start_time=20.0),
    "ais_only_initiation": result_summary(t5_ais_only, scenario_c, start_time=20.0),
    "rmse_available_windows_m": {
        "without_ais": round(
            np.mean([
                rmse(t5_no_ais, scenario_c, start_time=20.0, end_time=60.0),
                rmse(t5_no_ais, scenario_c, start_time=90.0),
            ]),
            3,
        ),
        "with_ais": round(
            np.mean([
                rmse(t5_with_ais, scenario_c, start_time=20.0, end_time=60.0),
                rmse(t5_with_ais, scenario_c, start_time=90.0),
            ]),
            3,
        ),
    },
    "dropout_60_90": {
        "rmse_without_ais_m": round(rmse(t5_no_ais, scenario_c, start_time=60.0, end_time=90.0), 3),
        "rmse_with_ais_m": round(rmse(t5_with_ais, scenario_c, start_time=60.0, end_time=90.0), 3),
        "max_history_gap_s": round(max_history_gap(t5_with_ais, 60.0, 90.0), 3),
        "accepted_radar_camera_updates": sum(
            1
            for row in t5_with_ais.history
            if 60.0 <= row.time <= 90.0 and ("radar" in row.sensors or "camera" in row.sensors)
        ),
    },
    "first_ais_reacquisition_after_90_s": first_accepted_after(t5_with_ais, "ais", 90.0),
}

print(json.dumps(t5_summary, indent=2))

{
  "without_ais": {
    "bootstrap_sensor": "radar",
    "bootstrap_time_s": 6.4,
    "confirmation_time_s": 35.2,
    "accepted_updates": 56,
    "rmse_after_start_m": 4.227,
    "nis_inside_95_pct": 86.44,
    "accepted_by_sensor": {
      "radar": 40,
      "camera": 19,
      "ais": 0
    }
  },
  "with_ais": {
    "bootstrap_sensor": "radar",
    "bootstrap_time_s": 3.2,
    "confirmation_time_s": 6.4,
    "accepted_updates": 85,
    "rmse_after_start_m": 3.042,
    "nis_inside_95_pct": 87.63,
    "accepted_by_sensor": {
      "radar": 41,
      "camera": 19,
      "ais": 37
    }
  },
  "ais_only_initiation": {
    "bootstrap_sensor": "ais",
    "bootstrap_time_s": 6.0,
    "confirmation_time_s": 12.0,
    "accepted_updates": 38,
    "rmse_after_start_m": 3.606,
    "nis_inside_95_pct": 86.84,
    "accepted_by_sensor": {
      "radar": 0,
      "camera": 0,
      "ais": 38
    }
  },
  "rmse_available_windows_m": {
    "without_ais": 4.478,
    "with_ais": 3.147
  },
  "dropout_

The AIS dropout is `60-90 s`. The tracker continues on radar/camera during that interval and accepts AIS again at the first AIS scan after the blackout.

## 6. Save Report

In [7]:
report = {
    "T2": t2_results,
    "T3": t3_summary,
    "T3_pass": t3_pass,
    "T4": t4_summary,
    "T5": t5_summary,
}

out_path = TRACKING_DIR / "results_t2_t5.json"
out_path.write_text(json.dumps(report, indent=2), encoding="utf-8")
print(f"Wrote {out_path}")

Wrote c:\Users\kryst\Documents\GitHub\Multi-Sensor-Multi-Target-Tracking-System-for-Harbour-Surveillance\tracking_solution\results_t2_t5.json
